# ReefScan — Parts C & D: self-hosted Qwen2.5-VL (vLLM) + serving sweep

**Standalone notebook — run on its own Colab GPU runtime (L4/A100).** Safe to run in parallel with the Part B notebook. Flow (one unavoidable restart, because the vLLM install swaps torch):
1. Run **cell 0** (clone + GPU), then the **install** cell.
2. **Runtime → Restart session.**
3. Re-run **cell 0**, then **C1 → C2 → C3 → D**.

Every number is printed by its cell; record the GPU from cell 0.

## 0. Clone repo + confirm GPU

In [ ]:
!git clone -q https://github.com/HrishiKabra/reefscan.git 2>/dev/null || echo 'repo already present'
%cd /content/reefscan
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU:', GPU, '| torch', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime -> Change runtime type -> L4/A100'

## C. Self-hosted VLM baseline via vLLM (Qwen2.5-VL-7B)

> **This install swaps torch → breaks torchvision.** After it + the restart, run ONLY C1→D; do not
> re-run Part B in this kernel (that's the `torchvision::nms` error — expected).

Replace the paid GPT-4o VLM baseline with a **self-hosted** open VLM served by vLLM's OpenAI-compatible
server, and re-run the exact same benchmark (forced A/B + logprobs → accuracy, macro-F1, ECE) on the same
NOAA test split. **GPT-4o stays as a column; we ADD Qwen.** The coral patches are 224px crops sent
unresized — Qwen2.5-VL sees them at native resolution.

Needs ~16 GB VRAM in fp16 (L4/A100 fine). **Pinned to vLLM 0.10.2** — 0.8.x is too old for
Qwen2.5-VL's rope config (`rope_type` vs legacy `type=mrope`), while the *latest* vLLM is a
CUDA-13 build (`libcudart.so.13`) that won't load on Colab's CUDA-12.8 runtime. 0.10.2 is a
cu128 wheel with the rope fix. (Fallbacks if needed: `0.9.2` if you see libcudart.so.13; `0.11.0` if the rope error returns.)

In [ ]:
!pip uninstall -q -y vllm
!pip install -q 'vllm==0.10.2'   # cu128 + Qwen2.5-VL rope fix; NOT `-U` (latest is cu13, breaks on Colab 12.8)
import vllm; print('vLLM', vllm.__version__)
print('>>> Now do Runtime -> Restart session, then re-run from the `%cd /content/reefscan` cell '
      'DOWN (skip git clone — it persists). The torch swap from this install needs a fresh kernel.')

### C1. Start the vLLM OpenAI-compatible server (backgrounded)

In [ ]:
import subprocess, sys, time, os, urllib.request
os.makedirs('logs', exist_ok=True)
srv = subprocess.Popen(
    [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
     '--model', 'Qwen/Qwen2.5-VL-7B-Instruct',
     '--gpu-memory-utilization', '0.9',
     '--max-model-len', '8192',      # leaves KV-cache headroom for higher concurrency
     '--max-logprobs', '20',         # A/B logprobs need top_logprobs <= this
     '--limit-mm-per-prompt', '{"image": 1}',   # vLLM 0.10.x wants JSON here, not image=1
     '--port', '8000'],
    stdout=open('logs/vllm.log','w'), stderr=subprocess.STDOUT)
print('starting vLLM (first run downloads ~16GB weights + warms up — can take several minutes)...')
up = False
for i in range(180):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); up = True; break
    except Exception: time.sleep(5)
print('vLLM server UP after ~%ds' % (i*5) if up else 'NOT up yet — inspect logs/vllm.log')
!tail -6 logs/vllm.log

### C2. Run the benchmark against the local server
Full test set (1,565 imgs) is resumable (JSONL cache) — start with `--limit 300` for a fast first pass,
then drop `--limit` for the full number. `--workers 16` drives concurrency at the server.

In [ ]:
!python -m backend.vlm_benchmark --model Qwen/Qwen2.5-VL-7B-Instruct \
    --base-url http://localhost:8000/v1 --api-key EMPTY --workers 16 --limit 300

### C3. Three-way comparison — DINOv2 specialist vs GPT-4o vs Qwen2.5-VL

In [ ]:
import json, glob
from pathlib import Path
E = Path('docs/eval')
bym = {}
spec = json.loads((E/'metrics.json').read_text())
bym['DINOv2-B specialist'] = (spec['accuracy'], spec['macro_f1'], spec.get('ece'))
# old committed gpt-4o summary + any new per-model files
cands = list(glob.glob(str(E/'vlm_benchmark_*.json')))
if (E/'vlm_benchmark.json').exists(): cands.append(str(E/'vlm_benchmark.json'))
for f in cands:
    d = json.loads(open(f).read())
    bym[d['model']] = (d['accuracy'], d['macro_f1'], d.get('ece'))
print(f"{'model':<32}{'accuracy':>9}{'macro-F1':>10}{'ECE':>8}")
print('-'*59)
for m,(a,f1,e) in bym.items():
    es = f'{e:.4f}' if e is not None else '  n/a'
    print(f'{m:<32}{a:>9.4f}{f1:>10.4f}{es:>8}')

**C result (fill from the run):** DINOv2-B specialist `<acc/F1/ECE>` vs zero-shot GPT-4o `<...>` vs
self-hosted Qwen2.5-VL-7B `<...>`. Expectation: the tiny fine-tuned specialist beats both generalist VLMs
on accuracy + calibration, and the self-hosted Qwen is a $0, reproducible stand-in for the paid GPT-4o
baseline (same prompt, same test split).

## D. vLLM serving throughput sweep (concurrency 1→32)

The serving curve for the self-hosted VLM: p50/p95/p99 latency + throughput as concurrency rises.
Primary path uses vLLM's built-in **`vllm bench serve`**; if the CLI/keys differ in your vLLM build, the
cell falls back to a small async sweep of the *actual* image workload against the chat endpoint so you
always get numbers. (No Docker/Triton in Colab — that's the RunPod appendix.)

In [ ]:
import json, subprocess, asyncio, io, time
import numpy as np
from PIL import Image
import httpx
LEVELS = [1,4,8,16,32]

def _pctile(v,p):
    if not v: return 0.0
    v=sorted(v); k=(len(v)-1)*p/100; f=int(k)
    return round(v[f]+(v[min(f+1,len(v)-1)]-v[f])*(k-f),1)

rows = []
# --- primary: vllm bench serve (text 'random' dataset; measures decode/serving throughput) ---
for c in LEVELS:
    rf = f'logs/bench_c{c}.json'
    subprocess.run(['vllm','bench','serve','--backend','openai-chat',
        '--model','Qwen/Qwen2.5-VL-7B-Instruct','--base-url','http://localhost:8000',
        '--endpoint','/v1/chat/completions','--dataset-name','random','--num-prompts','100',
        '--max-concurrency',str(c),'--percentile-metrics','e2el','--metric-percentiles','50,95,99',
        '--save-result','--result-filename',rf], check=False)
    try:
        d=json.load(open(rf))
        rows.append({'concurrency':c,'p50':d.get('median_e2el_ms') or d.get('p50_e2el_ms'),
                     'p95':d.get('p95_e2el_ms'),'p99':d.get('p99_e2el_ms'),
                     'throughput':d.get('request_throughput')})
    except Exception as ex:
        print('vllm bench serve produced no parseable result for c=%d (%s)'%(c,ex))

# --- fallback: async sweep of the real image workload if the CLI path yielded nothing ---
if not rows:
    print('falling back to async image-workload sweep...')
    arr=np.random.default_rng(0).integers(0,255,(224,224,3),dtype=np.uint8)
    buf=io.BytesIO(); Image.fromarray(arr).save(buf,format='PNG')
    import base64; b64=base64.b64encode(buf.getvalue()).decode()
    body=lambda: {'model':'Qwen/Qwen2.5-VL-7B-Instruct','max_tokens':1,'temperature':0,
        'messages':[{'role':'user','content':[{'type':'text','text':'A for healthy, B for bleached. One letter.'},
        {'type':'image_url','image_url':{'url':'data:image/png;base64,'+b64}}]}]}
    async def one(cl):
        t=time.perf_counter(); r=await cl.post('http://localhost:8000/v1/chat/completions',json=body()); r.raise_for_status()
        return (time.perf_counter()-t)*1000
    async def level(c,n=48):
        sem=asyncio.Semaphore(c); lat=[]
        async with httpx.AsyncClient(timeout=120) as cl:
            async def w():
                async with sem: lat.append(await one(cl))
            t=time.perf_counter(); await asyncio.gather(*[w() for _ in range(n)]); wall=time.perf_counter()-t
        return {'concurrency':c,'p50':_pctile(lat,50),'p95':_pctile(lat,95),'p99':_pctile(lat,99),
                'throughput':round(n/wall,2)}
    for c in LEVELS: rows.append(await level(c))

print(f"[{GPU}]  conc   p50 ms   p95 ms   p99 ms   req/s")
for r in rows:
    print(f"      {r['concurrency']:>4} {r['p50']:>8} {r['p95']:>8} {r['p99']:>8} {r['throughput']:>7}")
json.dump({'gpu':GPU,'rows':rows}, open('docs/eval/vllm_serving_sweep.json','w'), indent=2)

In [ ]:
import matplotlib.pyplot as plt
cs=[r['concurrency'] for r in rows]
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4.5))
for k,col in [('p50','#2166ac'),('p95','#b2182b'),('p99','#762a83')]:
    a1.plot(cs,[r[k] for r in rows],'-o',label=k,color=col)
a1.set_xlabel('concurrency'); a1.set_ylabel('latency (ms)'); a1.set_title(f'Qwen2.5-VL serving latency ({GPU})')
a1.legend(); a1.grid(alpha=.25)
a2.plot(cs,[r['throughput'] for r in rows],'-o',color='#1b7837')
a2.set_xlabel('concurrency'); a2.set_ylabel('throughput (req/s)'); a2.set_title('Throughput'); a2.grid(alpha=.25)
fig.tight_layout(); fig.savefig('docs/eval/vllm_serving_sweep.png',dpi=140,bbox_inches='tight'); plt.show()

**D result (fill from the run):** p99 pulls away from p50 as concurrency rises and throughput saturates —
the serving curve for a self-hosted 7B VLM on `<GPU>`. Download `docs/eval/vllm_serving_sweep.{json,png}`
and paste the table back to commit it.